# ISIC 2024 - Standalone Image Model Pipeline (Kaggle Run All)

This notebook is designed to train and run inference for a standalone image model on Kaggle.

### Steps to run on Kaggle:
1. **Add your Code Dataset**: Attach the dataset containing only your `image_only` folder (or the full repository).
2. **Add the Competition Dataset**: Attach the `isic-2024-challenge` dataset.
3. **Run All**: Ensure GPU accelerator is enabled (T4 or P100).

In [1]:
# =========================
# CELL 1: DATA & PATHS CONFIG
# =========================
from pathlib import Path
import sys

RUN_MODE = 'auto' # 'auto' | 'kaggle' | 'local'

# Kaggle Paths
KAGGLE_CODE_DATASET_ROOT = '/kaggle/input/datasets/trnhintmh03/isic-2024-v19' # Adjust this to your Kaggle Code dataset path
KAGGLE_COMP_DATASET_ROOT = '/kaggle/input/competitions/isic-2024-challenge' # Adjust if different
KAGGLE_OUTPUT_DIR = '/kaggle/working/image_only_outputs'

# Local Paths
LOCAL_CODE_DATASET_ROOT = r'D:\HK1_2025_2026\SEMINAR_CODE'
LOCAL_DATA_ROOT = r'D:\HK1_2025_2026\SEMINAR_CODE\data'
LOCAL_OUTPUT_DIR = './local_image_only_outputs'

is_kaggle = Path('/kaggle/input').exists() if RUN_MODE == 'auto' else (RUN_MODE == 'kaggle')

if is_kaggle:
    code_root = Path(KAGGLE_CODE_DATASET_ROOT)
    comp_root = Path(KAGGLE_COMP_DATASET_ROOT)
    TRAIN_METADATA_PATH = str(comp_root / 'train-metadata.csv')
    TEST_METADATA_PATH = str(comp_root / 'test-metadata.csv')
    SAMPLE_SUBMISSION_PATH = str(comp_root / 'sample_submission.csv')
    TRAIN_HDF5_PATH = str(comp_root / 'train-image.hdf5')
    TEST_HDF5_PATH = str(comp_root / 'test-image.hdf5')
    OUTPUT_DIR = Path(KAGGLE_OUTPUT_DIR)
else:
    code_root = Path(LOCAL_CODE_DATASET_ROOT)
    data_root = Path(LOCAL_DATA_ROOT)
    TRAIN_METADATA_PATH = str(data_root / 'train-metadata.csv')
    TEST_METADATA_PATH = str(data_root / 'test-metadata.csv')
    SAMPLE_SUBMISSION_PATH = str(data_root / 'sample_submission.csv')
    TRAIN_HDF5_PATH = str(data_root / 'train-image.hdf5')
    TEST_HDF5_PATH = str(data_root / 'test-image.hdf5')
    OUTPUT_DIR = Path(LOCAL_OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('✅ Running on:', 'KAGGLE' if is_kaggle else 'LOCAL')
print('✅ Code Root:', code_root)
print('✅ Output Directory:', OUTPUT_DIR)

✅ Running on: LOCAL
✅ Code Root: D:\HK1_2025_2026\SEMINAR_CODE
✅ Output Directory: local_image_only_outputs


In [2]:
# =========================
# CELL 1B: DEPENDENCIES & SYS PATH IMPORT SETUP
# =========================
import sys
from pathlib import Path

# Add the code dataset root directory to sys.path so we can import model, data, train modules directly
if str(code_root) not in sys.path:
    sys.path.insert(0, str(code_root))
print(f"✅ Added code root to sys.path: {code_root}")

# Find and install requirements
req_path = Path(code_root) / "requirements.txt"

if req_path.exists():
    print(f"✅ Installing requirements from: {req_path}")
    !pip -q install -r "{req_path}"
else:
    print("⚠️ requirements.txt not found. Installing key packages manually...")
    !pip -q install timm albumentations h5py

✅ Added code root to sys.path: D:\HK1_2025_2026\SEMINAR_CODE
⚠️ requirements.txt not found. Installing key packages manually...


In [3]:
# =========================
# CELL 2: HYPERPARAMETERS
# =========================
BACKBONE = 'efficientnet_b0'
IMAGE_SIZE = 224
PRETRAINED = False # Set to True if downloading from HuggingFace/timm via internet
BACKBONE_WEIGHTS_PATH = '/kaggle/input/datasets/trnhintmh03/model-backbone/efficientnet_b0_backbone.pth' # Set path to local weights (.pth) if offline
EPOCHS = 12
BATCH_SIZE = 64
LR = 4e-4
PATIENCE = 3
SEED = 42
FOLDS = 5
NEGATIVE_RATIO = 4
USE_AMP = True
NUM_WORKERS = 4 if is_kaggle else 0

print(f'✅ Backbone: {BACKBONE} | Image Size: {IMAGE_SIZE} | Epochs: {EPOCHS}')

✅ Backbone: efficientnet_b0 | Image Size: 224 | Epochs: 12


In [4]:
# =========================
# CELL 3: IMPORTS & PREPARATION
# =========================
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader

# Import standalone image model files directly
from mid_fusion_lite.image_only.model import StandaloneImageModel
from mid_fusion_lite.image_only.data import StandaloneImageDataset, get_image_transforms, make_balanced_sampler
from mid_fusion_lite.image_only.train import fit_one_fold, evaluate, predict_loader, partial_auc_score

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Using device: {device}')
print("✅ Successfully imported standalone image modules.")

c:\Users\khoan\miniconda3\envs\py311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Using device: cpu
✅ Successfully imported standalone image modules.


In [5]:
# =========================
# CELL 4: K-FOLD TRAINING
# =========================
df = pd.read_csv(TRAIN_METADATA_PATH)
id_col = 'isic_id'
target_col = 'target'
group_col = 'patient_id'

print(f'Loaded metadata size: {df.shape}')

# Split folds
sgkf = StratifiedGroupKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
df['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(sgkf.split(df, df[target_col], groups=df[group_col])):
    df.loc[val_idx, 'fold'] = fold

oof_preds = np.zeros(len(df), dtype=np.float32)
fold_scores = []
transforms = get_image_transforms(IMAGE_SIZE)

for fold in range(FOLDS):
    print(f'\n--- Training Fold {fold} ---')
    train_df = df[df['fold'] != fold].reset_index(drop=True)
    valid_df = df[df['fold'] == fold].reset_index(drop=True)
    
    train_dataset = StandaloneImageDataset(
        ids=train_df[id_col].to_numpy(),
        hdf5_path=TRAIN_HDF5_PATH,
        targets=train_df[target_col].to_numpy(),
        transforms=transforms['train']
    )
    
    valid_dataset = StandaloneImageDataset(
        ids=valid_df[id_col].to_numpy(),
        hdf5_path=TRAIN_HDF5_PATH,
        targets=valid_df[target_col].to_numpy(),
        transforms=transforms['valid']
    )
    
    sampler = make_balanced_sampler(
        targets=train_df[target_col].to_numpy(),
        negative_ratio=NEGATIVE_RATIO,
        seed=SEED + fold
    )
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        shuffle=(sampler is None),
        num_workers=NUM_WORKERS,
        pin_memory=True
    )
    
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )
    
    # Initialize model
    model = StandaloneImageModel(
        backbone_name=BACKBONE,
        use_pretrained=PRETRAINED,
        backbone_weights_path=BACKBONE_WEIGHTS_PATH
    )
    model = model.to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    # Fit
    best_state, history, best_score = fit_one_fold(
        model=model,
        train_loader=train_loader,
        valid_loader=valid_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        epochs=EPOCHS,
        patience=PATIENCE,
        pos_weight=1.0,
        use_amp=USE_AMP
    )
    
    checkpoint_path = OUTPUT_DIR / f'image_model_fold_{fold}.pt'
    torch.save(best_state, checkpoint_path)
    print(f'Saved Fold {fold} checkpoint to {checkpoint_path}')
    
    # Predict OOF
    model.load_state_dict(best_state)
    fold_val_preds = predict_loader(model=model, loader=valid_loader, device=device, use_amp=USE_AMP)
    
    val_indices = df[df['fold'] == fold].index
    oof_preds[val_indices] = fold_val_preds
    fold_scores.append(best_score)
    print(f'Fold {fold} pAUC Score: {best_score:.5f}')

C:\Users\khoan\AppData\Local\Temp\ipykernel_14864\3161432283.py:4: DtypeWarning: Columns (0: iddx_5, 1: mel_mitotic_index) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(TRAIN_METADATA_PATH)


Loaded metadata size: (401059, 55)


c:\Users\khoan\miniconda3\envs\py311\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)



--- Training Fold 0 ---


FileNotFoundError: Backbone weights file not found: /kaggle/input/datasets/trnhintmh03/model-backbone/efficientnet_b0_backbone.pth

In [ ]:
# =========================
# CELL 5: OOF METRIC EVALUATION
# =========================
oof_pauc = partial_auc_score(df[target_col].to_numpy(), oof_preds)
print(f'Overall Out-of-Fold pAUC: {oof_pauc:.5f}')
print(f'Mean Fold pAUC: {np.mean(fold_scores):.5f}')

# Save OOF CSV
df['pred'] = oof_preds
df[[id_col, target_col, 'fold', 'pred']].to_csv(OUTPUT_DIR / 'oof_predictions.csv', index=False)
print('Saved OOF predictions to oof_predictions.csv')

In [ ]:
# =========================
# CELL 6: TEST INFERENCE & SUBMISSION
# =========================
print('Running inference on test dataset...')
test_df = pd.read_csv(TEST_METADATA_PATH)
print(f'Test dataset size: {test_df.shape}')

transforms = get_image_transforms(IMAGE_SIZE)
test_dataset = StandaloneImageDataset(
    ids=test_df[id_col].to_numpy(),
    hdf5_path=TEST_HDF5_PATH,
    transforms=transforms['valid']
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

ensemble_preds = np.zeros(len(test_df), dtype=np.float32)

for fold in range(FOLDS):
    print(f'Predicting with Fold {fold} model...')
    model = StandaloneImageModel(backbone_name=BACKBONE)
    checkpoint_path = OUTPUT_DIR / f'image_model_fold_{fold}.pt'
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model = model.to(device)
    
    fold_preds = predict_loader(model=model, loader=test_loader, device=device, use_amp=USE_AMP)
    ensemble_preds += fold_preds / FOLDS

# Create Submission File
sub_df = pd.DataFrame({
    'isic_id': test_df[id_col],
    'target': ensemble_preds
})

# Save submission to current directory for Kaggle output matching
sub_df.to_csv('submission.csv', index=False)
print('✅ submission.csv successfully generated and saved!')
print(sub_df.head())